# Človek-v-zanki: Pre-koraki, razvrščanje tveganj in revizijsko beleženje

README za to lekcijo uvaja Človeka-v-zanki s kratkim odlomkom, ki uporabnika vpraša za `POTRJENJE` ali `ZAVRNITEV` potem, ko je agent že ustvaril odgovor. Ta vzorec je dober izhodiščni točka, vendar produkcijske implementacije HITL običajno potrebujejo tri dodatne dele:

1. **Pre-korak** ki teče **pred** izvajanjem tveganega koraka agenta, da ostanejo stroški, nepopravljivost in zakasnitev pod nadzorom.
2. **Razvrščanje tveganj**, tako da nizkotveganjska dejanja tečejo samodejno, srednje tvegana se potrjujejo v serijah, visoko tvegana pa zahtevajo človeško potrditev.
3. **Revizijski dnevnik in zanka revizij**, tako da je vsaka odločitev pre-koraka zabeležena kot JSONL, zavrnitev pa vzpodbuja agenta s strukturiranim razlogom namesto samo izpisa `Revidiranje...`.

Ta zvezek gradi vsak od teh elementov na istih primitivih kot `06-system-message-framework.ipynb`. Izvaja se od začetka do konca v `DEMO_MODE = True` (ni potrebno interaktivno vnašanje) ali z resničnimi `input()` pozivi, ko je `DEMO_MODE = False`. Opomba: v DEMO_MODE je ponoven poskus tretjega cilja skriptiran, da so mehanizmi zanke vidni od začetka do konca. Resnično preklasificiranje na podlagi revizije zahteva `DEMO_MODE = False` in operaterja.

**Izven obsega (obravnavano v drugih lekcijah):** avtentikacija in nadzor dostopa (Lekcija 06 README tveganje 2), posredovanje klicev orodij (Lekcija 14 MAF podrobno), vzorci večagentnih razprav.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Vzorec 1: Prehod pred dejanjem

Odsek HITL v README najprej pokliče agent, nato pa uporabnika zaprosi za odobritev izhoda. To je potek **po dejanju**. Agent je že izvedel, zato so stroški klica LLM že plačani, in vsak stranski učinek (poslano e-poštno sporočilo, zapisan vrstica v zbirki podatkov, objavljen komentar) se je že zgodil.

Potek **pred dejanjem** vstavi prehod pred agentom, ki izvede tvegani korak. Agent predlaga dejanje, prehod odloča, ali se naj izvede, in šele ob odobritvi pride do stranskega učinka.

| Vidik | Odobritev po dejanju (odsek README) | Prehod pred dejanjem (ta zvezek) |
|---|---|---|
| Kdaj se izvede odobritev? | Po tem, ko je agent ustvaril izhod | Pred izvajanjem katerega koli stranskega učinka |
| Strošek LLM pri zavrnitvi | Že plačan | Plačan samo za predlog, ne za dejanje |
| Nepovratni stranski učinki | Možni (dejanje se je že zgodilo) | Preprečeni |
| Jasnost revizije | Odobritev je izpisna izjava | Odobritev je JSON zapis s časovnim žigom, dejanjem, razlogom |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Vzorec 2: Razvrščanje tveganj po nivojih

Ne vsako dejanje zahteva človeško odobritev. Poizvedovanje samo za branje preko javnega vmesnika API ima drugačne vložke kot pošiljanje e-pošte stranki. Enako obravnavanje obeh pomeni izgubo pozornosti upravljavca in upočasni agent.

Preprost model s 3 nivoji:

| Nivo | Primeri | Tok odobritve |
|---|---|---|
| `nizka` (samo za branje) | Iskanje v podatkovni bazi znanja, pregled letalskih možnosti, pridobivanje javne spletne strani | Samodejno izvedeno, zabeleženo za revizijo |
| `srednja` (cenovno ugodna mutacija) | Predpomnilnik rezultata, povečanje števca, načrtovanje opomnika | Samodejno izvedeno, a združeno za dnevni pregled |
| `visoka` (usmerjeno navzven ali nepovratno) | Pošiljanje e-pošte, zaračunavanje kartice, objava v javnem kanalu | Blokirano do človeške odobritve |

To je eno razvrščanje nivojev. Produkcijski sistemi pogosto uporabljajo bolj natančne nivoje (npr. ravni dovoljenj AWS IAM, nivoji dostopa na podlagi vlog). Verzija s 3 nivoji spodaj je najmanjša uporabna različica za agenta, ki meša dejanja samo za branje in tista z neželenimi učinki.

Razvrščevalnik spodaj uporablja heuristiko ključnih besed, da demo ostane determinističen in poceni. V produkcijskem sistemu bi to zamenjali z naučenim razvrščevalnikom ali sistemom pravil.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzorec 3: Revizijska dnevniška datoteka in zanka revizije

`print("Response approved.")` ni revizijska dnevniška datoteka. Za zaupanje je treba vsako odločitev portala zabeležiti kot strukturiran dogodek, ki ga lahko pozneje poizvedete, predvajate ali priložite pregledu incidenta.

Dve sestavini:

1. **Samo-dodajalen JSONL.** Ena vrstica na odločitev s časovnim žigom, dejanjem, nivojem, odločitvijo, razlogom. Enostavno za grep, enostavno kasneje poslati v resnično dnevniško shrambo.
2. **Zanka revizije ob zavrnitvi.** Ko portal vrne `deny`, si agent znova zastavi vprašanje z razlogom zavrnitve v kontekstu, da lahko naslednji predlog prepreči težavo.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Dodatni viri

Več drugih javnih projektov izvaja različice teh HITL vzorcev. Primerjajte pristope, da najdete tistega, ki ustreza vašemu stacku:

- **LangChain** ovojnice orodij human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ovojnice orodij, ki začasno zaustavijo izvajanje za človeški vnos.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ je to prestrukturiral): uporablja vlogo agenta, ki posebej predstavlja človeka v pogovorih več agentov.
- **Semantic Kernel** filtri funkcij ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, ki teče okoli vsakega klica funkcije, primeren za usmerjanje logike.

Vsak projekt drugače obravnava tri pod-vzorce: LangChain jih ovije kot orodja, AutoGen uporablja vlogo agenta, Semantic Kernel uporablja middleware filtre. Preberite eno ali dve implementaciji od začetka do konca, preden izberete zasnovo za svojega agenta.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
